# Feature-population, stratified-null and per-feature utility audit

Completed results use the historical 37 validation patients. NCL/SimCLR baseline
SAE states are archived; nine supervised-reference SAEs were newly fitted on saved
supervised embeddings. This notebook does not retrain encoders or access test images.

Use the **full authorized audit artifact** for input vectors/model states. The
code-only ZIP is insufficient to reproduce numerical outputs without those assets.


In [ ]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd
from IPython.display import display, Image
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p/'scripts/feature_population_audit.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the extracted audit artifact folder.")
OUT = ROOT/'results/feature_population_audit_v1'
print("Inputs and results:", OUT)


## Optional full audit replay

The published outputs are already included. Set `RECOMPUTE_AUDITS=True` to repeat
all 27 fixed-model audits (GPU recommended). This recomputes training-oriented
univariate AUROCs, 1,000 global and stratified nulls and 1,000 random feature subsets.
SAE and encoder fitting is not repeated. The fixed protocol must match.


In [ ]:
RECOMPUTE_AUDITS = False
if RECOMPUTE_AUDITS:
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/feature_population_audit.py'),
                    '--methods', 'ncl', 'simclr', 'supervised', '--force-recompute'], cwd=ROOT, check=True)
    subprocess.run([sys.executable, str(ROOT/'work/report_feature_population_audit.py')], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, str(ROOT/'scripts/verify_feature_population_audit.py')], cwd=ROOT, check=True)
print((OUT/'VERIFICATION.json').read_text())


In [ ]:
per_encoder = pd.read_csv(OUT/'per_encoder.csv')
display(per_encoder.groupby('method')[['rho_purity_utility', 'rho_stratified_excess_utility',
    'mean_purity', 'mean_global_excess', 'mean_stratified_excess']].agg(['mean', 'std']))
display(Image(filename=str(OUT/'feature_budget_audit.png')))


## Interpretation

- Raw purity and individual-feature AUROC have positive correlations here.
- Area/index-stratified excess has near-zero mean correlation with univariate AUROC.
- The all-eligible NCL advantage is absent under the primary stratified reference.
- Budget curves and random-subset references expose selection-rule dependence.
- Beyond the fully replicated common budget, individual dictionary endpoints are
  not equal-size method comparisons. Missing-support bounds are not confidence intervals.
- The stratified null conditions on annotation-derived area, including tumour absence;
  it is descriptive and does not establish causality or clinical explanation validity.
- Means average SAE seeds within encoder; only three independent encoders per objective.


In [ ]:
# Optional quick oracle-control replay; no model training or patient bootstrap.
subprocess.run([sys.executable, str(ROOT/"scripts/check_stratified_null_sensitivity.py")], cwd=ROOT, check=True)
display(pd.read_csv(OUT/"null_sensitivity_controls.csv"))


## Paired whole-patient bootstrap for Fig. 2B
Already computed: 2,000 paired draws, fixed models and feature IDs. The band includes missing-support bounds; it is conditional and pointwise, not a simultaneous confidence band.


In [ ]:
REPLAY_PATIENT_BOOTSTRAP = False
if REPLAY_PATIENT_BOOTSTRAP:
    subprocess.run([sys.executable, str(ROOT/"scripts/bootstrap_feature_budget_patients.py")], cwd=ROOT, check=True)
display(pd.read_csv(OUT/"patient_budget_bootstrap/curve_intervals.csv").query("J in [8,32,64,80]"))


## Patient uncertainty for the null and feature correlations
Paired whole-patient bootstrap, fixed models/orientations, recomputed top tens and conditional null. The all-eligible rule is reapplied per draw; this differs from fixed-ID budget bounds. This remains exploratory validation uncertainty.


In [ ]:
REPLAY_NULL_CORRELATION_BOOTSTRAP = False
if REPLAY_NULL_CORRELATION_BOOTSTRAP:
    subprocess.run([sys.executable, str(ROOT/"scripts/bootstrap_null_correlations.py")], cwd=ROOT, check=True)
display(pd.read_csv(OUT/"patient_null_correlation_bootstrap/intervals.csv"))
